# 04 — Modelo predictivo

Score compuesto interpretable + validación con regresión logística y **backtest temporal** (sin fuga: features solo con pasado, test en meses posteriores al corte).

In [ ]:
# Ejecutar desde la raiz del repo (o ajustar la ruta)
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

import pandas as pd
pd.set_option("display.max_columns", 40)

In [ ]:
from src.common import DATA_PROCESSED
from src.features.build_features import construir_panel
from src.train import backtest, _params

sol = pd.read_csv(DATA_PROCESSED / "vitales" / "solicitudes.csv")
panel = construir_panel(sol)
print(f"Panel: {len(panel):,} filas ({panel['pa'].nunique()} principios activos x {panel['mes'].nunique()} meses)")

In [ ]:
params = _params()
resultados, ultimo = backtest(panel, params["modelo"]["cortes_backtest"],
                              params["modelo"]["label_horizonte_meses"],
                              params["modelo"]["precision_at_k"])
pd.DataFrame(resultados)

In [ ]:
# Metricas agregadas
aucs = [r["auc"] for r in resultados]
p20 = [r["precision_at_20"] for r in resultados]
print(f"AUC promedio: {sum(aucs)/len(aucs):.3f} | precision@20 promedio: {sum(p20)/len(p20):.3f}")

## Score compuesto (interpretable por diseño)

In [ ]:
from src.inference import calcular_scores
scores = calcular_scores(panel)
scores.head(10)[["principio_activo", "score", "nivel", "tendencia"]]

In [ ]:
scores["nivel"].value_counts()

In [ ]:
from IPython.display import Image
Image("../reports/figures/matriz_confusion.png")

In [ ]:
from IPython.display import Image
Image("../reports/figures/roc.png")